# Notebook 03 — Обучение модели PilotNet

## Что делаем:
1. Загружаем архитектуру PilotNet и выводим summary
2. Создаём DataLoader
3. Запускаем обучение с EarlyStopping
4. Смотрим кривые loss
5. Сохраняем лучшую модель на Google Drive

---
> ⚠️ **Убедись что выбран GPU:** `Среда выполнения → Сменить тип среды → GPU (T4)`

In [6]:
!pip install -q torch torchvision numpy pandas matplotlib opencv-python scikit-learn tqdm

from google.colab import drive
drive.mount('/content/drive')

import sys, os, json
import torch
import numpy as np

# Загружаем конфигурацию
with open('/content/drive/MyDrive/diploma/config.json') as f:
    cfg = json.load(f)

PROJECT_DIR = cfg['project_dir']
DATA_DIR    = cfg['data_dir']
MODELS_DIR  = cfg['models_dir']
OUTPUTS_DIR = cfg['outputs_dir']
CSV_PATH    = cfg['csv_path']

# Добавляем src/ в путь
src_path = os.path.join(PROJECT_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Воспроизводимость
torch.manual_seed(42)
np.random.seed(42)

# Устройство
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
    print(f'  Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ GPU не найден, используется CPU — обучение будет медленнее')
    print('  Перейди: Среда выполнения → Сменить тип среды → GPU')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚠ GPU не найден, используется CPU — обучение будет медленнее
  Перейди: Среда выполнения → Сменить тип среды → GPU


## Шаг 1. Архитектура модели

In [7]:
from model import get_model

# Создаём модель
model = get_model(dropout_rate=0.5, device=device)

# Выводим архитектуру и количество параметров
model.summary(device=device)

       АРХИТЕКТУРА NVIDIA PilotNet
PilotNet(
  (conv_layers): Sequential(
    (0): Conv2d(3, 24, kernel_size=(5, 5), stride=(2, 2))
    (1): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ELU(alpha=1.0)
    (3): Conv2d(24, 36, kernel_size=(5, 5), stride=(2, 2))
    (4): BatchNorm2d(36, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ELU(alpha=1.0)
    (6): Conv2d(36, 48, kernel_size=(5, 5), stride=(2, 2))
    (7): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ELU(alpha=1.0)
    (9): Conv2d(48, 64, kernel_size=(3, 3), stride=(1, 1))
    (10): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): ELU(alpha=1.0)
    (12): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (13): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ELU(alpha=1.0)
  )
  (fc_layers): Sequential(
    (0): Flatten(start_dim=1, e

## Шаг 2. Подготовка данных

In [8]:
from dataset import get_dataloaders

train_loader, val_loader, test_loader = get_dataloaders(
    csv_path=CSV_PATH,
    data_dir=os.path.dirname(CSV_PATH),
    batch_size=32,
    balance=True,
    num_workers=0,  # 0 для стабильной работы в Colab
)

# Сохраняем test_loader для notebook 04
# (он создаётся заново там из того же CSV)
print('✓ DataLoader готов')

Загружено строк из CSV: 3930
Итого сэмплов (×3 камеры): 11790
После балансировки: 3127 сэмплов (было 11790)

Разбивка датасета:
  Train: 2188 сэмплов
  Val:   469  сэмплов
  Test:  470 сэмплов
✓ DataLoader готов


## Шаг 3. Обучение

**Параметры обучения:**
- Оптимизатор: Adam (lr=1e-4, weight_decay=1e-4)
- Loss: MSE (Mean Squared Error)
- Планировщик: ReduceLROnPlateau (×0.5 при стагнации 5 эпох)
- EarlyStopping: patience=10 эпох
- Максимум эпох: 50

In [9]:
from train import train as train_model

history = train_model(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    num_epochs   = 50,
    lr           = 1e-4,
    patience     = 10,
    save_dir     = MODELS_DIR,
    device       = device,
)

TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

## Шаг 4. Анализ результатов обучения

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

train_losses = history['train_losses']
val_losses   = history['val_losses']
best_epoch   = history['best_epoch']
best_val_loss = history['best_val_loss']

print(f'Обучение завершено!')
print(f'  Лучшая эпоха:   {best_epoch}')
print(f'  Лучший val MSE: {best_val_loss:.6f}')
print(f'  Всего эпох:     {len(train_losses)}')

# Детальный график
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(train_losses) + 1)

# Обычный масштаб
ax = axes[0]
ax.plot(epochs, train_losses, 'b-o', markersize=3, label='Train Loss')
ax.plot(epochs, val_losses,   'r-o', markersize=3, label='Val Loss')
ax.axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7,
           label=f'Лучшая эпоха: {best_epoch}')
ax.set_xlabel('Эпоха'); ax.set_ylabel('MSE Loss')
ax.set_title('Кривые обучения (линейный масштаб)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

# Логарифмический масштаб (лучше виден прогресс)
ax = axes[1]
ax.semilogy(epochs, train_losses, 'b-o', markersize=3, label='Train Loss')
ax.semilogy(epochs, val_losses,   'r-o', markersize=3, label='Val Loss')
ax.axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7)
ax.set_xlabel('Эпоха'); ax.set_ylabel('MSE Loss (log)')
ax.set_title('Кривые обучения (лог. масштаб)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'training_curves.png'), dpi=150)
plt.show()

In [ ]:
# Проверяем что модель сохранена
model_path = os.path.join(MODELS_DIR, 'best_model.pth')
if os.path.exists(model_path):
    size_mb = os.path.getsize(model_path) / 1e6
    print(f'✓ Лучшая модель сохранена: {model_path}')
    print(f'  Размер файла: {size_mb:.1f} MB')
else:
    print('✗ Файл модели не найден!')

# Сохраняем историю обучения
import json
history_path = os.path.join(MODELS_DIR, 'training_history.json')
with open(history_path, 'w') as f:
    json.dump({
        'train_losses':  train_losses,
        'val_losses':    val_losses,
        'best_epoch':    best_epoch,
        'best_val_loss': best_val_loss,
    }, f, indent=2)
print(f'✓ История обучения: {history_path}')

## Итог

✅ Модель PilotNet обучена  
✅ EarlyStopping предотвратил переобучение  
✅ Лучшие веса сохранены в `models/best_model.pth`  
✅ Кривые обучения сохранены  

**Следующий шаг:** `04_evaluation.ipynb` — детальная оценка качества модели